# 02 — Data preparation, integration and databases

This notebook makes all integration decisions auditable: matching keys, duplicate handling, plausibility filters, weather join coverage and row changes.

In [ ]:
import pandas as pd
from IPython.display import display
from sptdelays.settings import PATHS
from sptdelays.prepare import prepare_transport, integrate_weather
from sptdelays.collect_weather import collect_weather
from sptdelays.database import build_sqlite
from sptdelays.duckdb_store import build_duckdb
from sptdelays.quality import run_quality_audit

## Prepare transport observations
The primary design uses repeated group-collected `live` snapshots across the approved study window. `actuals` is an optional historical extension. A single live snapshot is development evidence only. Delay is clipped at zero for the main outcome; signed delay and every exclusion remain auditable.

In [ ]:
SOURCE = 'live'  # repeated snapshots are the primary study design
transport = prepare_transport(SOURCE)
display(transport.head())
display(pd.read_csv(PATHS.tables / 'preparation_audit.csv'))

## Collect and integrate hourly weather
Weather is requested at each station coordinate and joined on station ID plus scheduled UTC hour. Local Europe/Zurich time is used for calendar/peak-period features. The left join must preserve every transport row; missing weather is recorded.

In [ ]:
REFRESH_WEATHER = False  # existing inputs make Run All reproducible without API calls
weather = collect_weather() if REFRESH_WEATHER else pd.read_csv(PATHS.interim / 'weather_hourly.csv')
model_data = integrate_weather()
display(pd.read_csv(PATHS.tables / 'integration_audit.csv'))

## SQLite and SQL from Python
The generated database contains normalized station, weather and observation tables, indexes and documented aggregation queries. Query outputs are exported to `reports/tables/`.

In [ ]:
database_path = build_sqlite()
duckdb_path = build_duckdb()
audit = run_quality_audit()
display(pd.read_csv(PATHS.tables / 'quality_checks.csv'))
database_path, duckdb_path

In [ ]:
display(pd.read_csv(PATHS.tables / 'sqlite_delay_by_region_mode.csv').head(20))
display(pd.read_csv(PATHS.tables / 'sqlite_weather_delay_join.csv').head(20))

## Final validation
Explain every lost row, report the weather match rate, inspect unmatched records by region/mode and verify that the final dataset covers the intended dates. Validated DuckDB evidence is produced with `sptdelays duckdb`; the full PostgreSQL option uses `sptdelays postgres`.